# 04b — Refinar F3 com lógica corrigida

**Notebook patch** — roda APENAS a auditoria F3 do SEEG após o fix da lógica `classify()` em `pipeline/seeg.py`.

**O que mudou:**
- Antes: `valor=0` era classificado como "OK" (porque `pd.notna(0) == True`)
- Depois: distingue 3 estados — `NaN`, `valor=0`, `valor>0` — e classifica em 6 categorias granulares

**Categorias novas:**
- `OK` — canavieiro com observação > 0
- `OK_ZERO_CANAVIEIRO` — canavieiro com valor 0 (registro estrutural sem emissão real)
- `LACUNA_SUSPEITA` — canavieiro com NaN (bug de cobertura SEEG)
- `RUIDO_ESTRUTURAL` — não-canavieiro × canal cana × valor > 0 (ruído do SEEG)
- `ZERO_LEGITIMO` — não-canavieiro × canal cana × valor 0 ou NaN
- `ZERO_NAO_CANAVIEIRO` — não-canavieiro × canal não-cana × valor 0

Não reroda o painel SEEG nem o PAM — apenas reaplica a F3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path
BASE_DIR = Path('/content/drive/MyDrive/Renovabio - EcoEco')
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Reload seeg com nova lógica
import importlib
from pipeline import seeg, pam
importlib.reload(seeg)
importlib.reload(pam)

from pipeline.config import interim, out_pre
from pipeline.pam import rerun_seeg_coverage_with_pam

Mounted at /content/drive


In [ ]:
# Carrega canavieiros baseline do PAM
canavieiros = pd.read_csv(interim('pam_canavieiro_baseline.csv'), dtype={'geocode': str})
print(f'Canavieiros PAM: {len(canavieiros)}')

# Reroda F3 com nova lógica
coverage_v3 = rerun_seeg_coverage_with_pam(
    canavieiros['geocode'].tolist(),
    save=True
)
print(f'\nF3 com nova lógica:')
print(coverage_v3['classificacao'].value_counts().to_string())

In [ ]:
# Análise do RUIDO_ESTRUTURAL: quantos munis não-canavieiros têm queima > 0?
ruido = coverage_v3[coverage_v3['classificacao'] == 'RUIDO_ESTRUTURAL']
if len(ruido) > 0:
    print(f'Cells RUIDO_ESTRUTURAL: {len(ruido):,}')
    print(f'Municípios distintos: {ruido["geocode"].nunique():,}')
    print(f'Por canal:')
    print(ruido.groupby('canal').size().to_string())
    print(f'\nMagnitude da queima nos cells de ruído (deve ser pequena):')
    print(ruido['valor'].describe())

In [ ]:
# Análise das LACUNAS_SUSPEITAS — canavieiros sem dado SEEG (bug real?)
lacunas = coverage_v3[coverage_v3['classificacao'] == 'LACUNA_SUSPEITA']
if len(lacunas) > 0:
    print(f'Cells LACUNA_SUSPEITA: {len(lacunas):,}')
    print(f'Municípios distintos: {lacunas["geocode"].nunique():,}')
    print(f'Por canal:')
    print(lacunas.groupby('canal').size().to_string())
    print(f'\nPor UF:')
    print(lacunas.groupby('uf')['geocode'].nunique().to_string())
else:
    print('✅ Nenhuma lacuna suspeita — cobertura SEEG completa para canavieiros PAM')

## Próximos passos

Se as classificações fizerem sentido (RUIDO_ESTRUTURAL com magnitude pequena, LACUNA_SUSPEITA pequeno ou nulo), F3 está validada.

Próxima camada: `05_mapbiomas.ipynb` — uso do solo, refina universo canavieiro com `mb_share_cana > 5%` (critério complementar §3.3).